# Stage 04: Localised (span-level) PCL detection (Part 3)

Utilises task2 dataset span text to get pure localised signals of where in paragh pcl is actually occuring, so the model can aggreagate these local signals and global context for more accurate paragraph-level predictions

## Imports & Dataset utilities

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import os
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

# project root (one level above notebooks)
ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

from src.data.make_dataset import build_task1_task2_with_spans, validate_span_ranges, validate_span_text_alignment, truncation_rate_by_label

RAW_TASK1 = ROOT / "data" / "raw" / "dontpatronizeme_pcl.tsv"
RAW_TASK2 = ROOT / "data" / "raw" / "dontpatronizeme_categories.tsv"
TRAIN_SPLIT = ROOT / "data" / "splits" / "train_semeval_parids-labels.csv"
DEV_SPLIT = ROOT / "data" / "splits" / "dev_semeval_parids-labels.csv"

train_df, dev_df, pcl_df, spans_df_norm = build_task1_task2_with_spans(
    raw_task1_path=RAW_TASK1,
    raw_task2_path=RAW_TASK2,
    train_split_path=TRAIN_SPLIT,
    dev_split_path=DEV_SPLIT,
)

print("train_df:", train_df.shape, "| positives:", int(train_df["label_bin"].sum()))
print("dev_df:", dev_df.shape, "| positives:", int(dev_df["label_bin"].sum()))
print("Example span_ranges:", train_df.loc[train_df["label_bin"].idxmax(), "span_ranges"] if len(train_df) else None)

/home/joshua_killa/.pyenv/versions/pcl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train_df: (8375, 8) | positives: 794
dev_df: (2094, 8) | positives: 199
Example span_ranges: [(157, 242)]


In [2]:
# Make sure span ranges are valid
validate_span_ranges(train_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="train")
validate_span_ranges(dev_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="dev")

# Check task2 spans match the corresponding substrings in task1 (only 2 mismatches and only by one charcter offset which is acceptable given token based stuff)
validate_span_text_alignment(
    train_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="train", max_mismatches=1
)
validate_span_text_alignment(
    dev_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="dev", max_mismatches=1
)


== validate_span_ranges: train ==
shape: (8375, 8)
missing (par_id,s,e) pairs: 0
out-of-bounds (par_id,s,e,L): 0
negatives with spans: 0

== validate_span_ranges: dev ==
shape: (2094, 8)
missing (par_id,s,e) pairs: 0
out-of-bounds (par_id,s,e,L): 0
negatives with spans: 0

== validate_span_text_alignment: train ==
total spans checked: 2466
text mismatches: 1
examples:


,par_id,span_start_norm,span_finish_norm,span_text,task1_substr
1134,4655,16,114,"To suffer and empathize together with others ,...","o suffer and empathize together with others , ..."



== validate_span_text_alignment: dev ==
total spans checked: 714
text mismatches: 1
examples:


,par_id,span_start_norm,span_finish_norm,span_text,task1_substr
482,6708,0,133,I only wish they can one day wake up to realis...,I only wish they can one day wake up to relise...


In [3]:
# ---- choose backbone here ----
MODEL_CANDIDATES = {
    "deberta": "microsoft/deberta-v3-base",
    "albert": "albert-base-v2",
    "albert_large": "albert-large-v2",
}

MODEL_KEY = os.getenv("PCL_BACKBONE", "albert_large")  # set to "albert" to try ALBERT
MODEL_NAME = MODEL_CANDIDATES[MODEL_KEY]
MAX_LEN = 192

def load_tokenizer(model_name: str):
    # Online first, then local cache fallback (for DNS/no-internet issues)
    try:
        return AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as e:
        print(f"Tokenizer load failed ({type(e).__name__}: {e}). Trying local cache...")
        return AutoTokenizer.from_pretrained(model_name, use_fast=True, local_files_only=True)

tokenizer = load_tokenizer(MODEL_NAME)

class PCLTokenDataset(Dataset):
    """
    Expects df columns:
      - text (str)
      - label_bin (0/1)
      - span_ranges (list[(start,end)])  (can be empty list)
    Produces:
      - input_ids, attention_mask
      - token_type_ids (only if tokenizer returns it; needed for ALBERT/BERT-like)
      - token_labels (0/1 per token)
      - token_loss_mask (bool mask for real tokens only; excludes padding + specials)
      - paragraph_label (float)
    """
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = row["text"]

        # ---- guard: tokenizer requires str / list[str] ----
        # If text is NaN/None/float, turn into empty string (or str(text) if you prefer).
        if text is None:
            text = ""
        elif isinstance(text, float):
            # catches NaN too (np.nan is float)
            if np.isnan(text):
                text = ""
            else:
                text = str(text)
        elif not isinstance(text, str):
            text = str(text)

        enc = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        offsets = enc["offset_mapping"][0]          # (T,2)
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]  # (T,)
        token_type_ids = enc["token_type_ids"][0] if "token_type_ids" in enc else None

        token_labels = torch.zeros(MAX_LEN, dtype=torch.float32)

        # real tokens: not padding AND not special tokens (specials often have offset (0,0))
        is_real_token = (attention_mask == 1) & (offsets[:, 1] > offsets[:, 0])

        spans = row["span_ranges"] if "span_ranges" in row.index else []
        if not isinstance(spans, list):
            spans = []

        # assign token label = 1 if token overlaps any annotated span
        for i, (start, end) in enumerate(offsets.tolist()):
            if not bool(is_real_token[i].item()):
                continue
            for s, e in spans:
                if start < e and end > s:
                    token_labels[i] = 1.0
                    break

        enc.pop("offset_mapping")

        batch = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_labels": token_labels,
            "token_loss_mask": is_real_token,
            "paragraph_label": torch.tensor(float(row["label_bin"]), dtype=torch.float32),
        }
        if token_type_ids is not None:
            batch["token_type_ids"] = token_type_ids

        return batch

In [4]:
# Verify max token length covers most examples (192 seems fine)
print(truncation_rate_by_label(tokenizer, train_df, 192))
print(truncation_rate_by_label(tokenizer, train_df, 200))
print(truncation_rate_by_label(tokenizer, train_df, 224))

{'max_len': 192, 'all_truncated_pct': 0.6447761194029851, 'pos_truncated_pct': 0.5037783375314862, 'neg_truncated_pct': 0.6595435958316844, 'pos_p99_len': 175, 'neg_p99_len': 180}
{'max_len': 200, 'all_truncated_pct': 0.4895522388059702, 'pos_truncated_pct': 0.5037783375314862, 'neg_truncated_pct': 0.4880622609154465, 'pos_p99_len': 175, 'neg_p99_len': 180}
{'max_len': 224, 'all_truncated_pct': 0.20298507462686569, 'pos_truncated_pct': 0.3778337531486146, 'neg_truncated_pct': 0.18467220683287164, 'pos_p99_len': 175, 'neg_p99_len': 180}


In [5]:
# How to pool token-level logits to single paragraph level logit (used in TokenCLSModel)

class LogitPooler(nn.Module):
    """
    Pool token-level logits (B,T) -> (B,1) using a boolean mask (B,T).

    Modes:
      - "max": masked max over tokens
      - "topk_mean": mean of top-k masked logits (k set by top_k)
    """
    def __init__(self, mode: str = "max", top_k: int = 3, sentinel: float = -1e4):
        super().__init__()
        self.mode = str(mode)
        self.top_k = int(top_k)
        self.sentinel = float(sentinel)

        if self.mode not in {"max", "topk_mean"}:
            raise ValueError(f"Unknown pooling mode: {self.mode}")

    def forward(self, token_logits: torch.Tensor, token_mask: torch.Tensor) -> torch.Tensor:
        """
        token_logits: (B,T) float
        token_mask:   (B,T) bool (True = keep / real tokens)
        returns:      (B,1)
        """
        token_mask = token_mask.to(dtype=torch.bool)

        # Edge-case guard: if a sample has no real tokens, return 0.0 (neutral feature)
        no_real = ~token_mask.any(dim=1, keepdim=True)  # (B,1)

        x = token_logits.masked_fill(~token_mask, self.sentinel)  # (B,T)

        if self.mode == "max":
            pooled = x.max(dim=1, keepdim=True).values  # (B,1)
            pooled = pooled.masked_fill(no_real, 0.0)
            return pooled

        # self.mode == "topk_mean"
        B, T = x.shape
        k = min(self.top_k, T)
        topk_vals = torch.topk(x, k=k, dim=1).values  # (B,k)

        # If k > #real tokens, topk will include sentinel values; exclude them from the mean.
        valid = topk_vals > (self.sentinel + 1.0)
        denom = valid.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (topk_vals.masked_fill(~valid, 0.0).sum(dim=1, keepdim=True)) / denom
        pooled = pooled.masked_fill(no_real, 0.0)
        return pooled

In [6]:
class TokenCLSModel(nn.Module):
    def __init__(self, model_name, lambda_token=0.3, pool_mode="max", top_k=3):
        super().__init__()

        # Online first, then local cache fallback
        try:
            self.encoder = AutoModel.from_pretrained(model_name)
        except Exception as e:
            print(f"Model load failed ({type(e).__name__}: {e}). Trying local cache...")
            self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)

        hidden = self.encoder.config.hidden_size

        self.token_head = nn.Linear(hidden, 1)
        self.paragraph_head = nn.Linear(hidden + 1, 1)

        self.lambda_token = float(lambda_token)

        # modular pooling (swap "max" <-> "topk_mean")
        self.pooler = LogitPooler(mode=pool_mode, top_k=top_k)

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
        token_labels=None,
        token_loss_mask=None,
        paragraph_label=None,
    ):
        # Some backbones ignore token_type_ids; some accept it.
        # Try passing it; if unsupported, fall back.
        try:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        except TypeError:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        hidden = outputs.last_hidden_state  # (B,T,H)

        # keep dtype consistent with heads (prevents dtype mismatch issues)
        hidden = hidden.to(dtype=self.token_head.weight.dtype)

        cls = hidden[:, 0]  # (B,H)
        token_logits = self.token_head(hidden).squeeze(-1)  # (B,T)

        # safe token mask for pooling + token-loss
        if token_loss_mask is None:
            token_loss_mask = attention_mask == 1
        token_loss_mask = token_loss_mask.to(dtype=torch.bool)

        pooled = self.pooler(token_logits, token_loss_mask)  # (B,1)

        fused = torch.cat([cls, pooled], dim=1)  # (B,H+1)
        paragraph_logit = self.paragraph_head(fused).squeeze(-1)  # (B,)

        loss = None
        loss_par = None
        loss_tok = None

        if paragraph_label is not None:
            # Compute BCE losses in FP32 for stability (important under AMP)
            loss_par = F.binary_cross_entropy_with_logits(
                paragraph_logit.float(),
                paragraph_label.float(),
            )

            tok_logits_flat = token_logits[token_loss_mask]
            if tok_logits_flat.numel() == 0:
                loss_tok = torch.zeros((), device=token_logits.device, dtype=torch.float32)
            else:
                tok_labels_flat = token_labels[token_loss_mask].float()
                loss_tok = F.binary_cross_entropy_with_logits(
                    tok_logits_flat.float(),
                    tok_labels_flat,
                )

            loss = loss_par + self.lambda_token * loss_tok

        return {
            "loss": loss,
            "loss_par": loss_par,
            "loss_tok": loss_tok,
            "paragraph_logit": paragraph_logit,
            "token_logits": token_logits,
        }

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import torch
import numpy as np
from sklearn.metrics import f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _amp_settings(model_key: str, model_name: str):
    key = (model_key or "").lower()
    name = (model_name or "").lower()

    if ("albert" in key) or ("albert" in name):
        enabled = torch.cuda.is_available()
        return enabled, torch.float16, True  # FP16 + GradScaler

    if ("deberta" in key) or ("deberta" in name):
        enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        return enabled, torch.bfloat16, False  # BF16, no GradScaler

    return False, None, False

USE_AMP, AMP_DTYPE, NEEDS_SCALER = _amp_settings(MODEL_KEY, MODEL_NAME)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

print(
    f"DEVICE={DEVICE} | backbone={MODEL_KEY} | amp={USE_AMP} | amp_dtype={AMP_DTYPE} | "
    f"scaler={bool(scaler.is_enabled())}"
)

model = TokenCLSModel(MODEL_NAME, lambda_token=0.3, mode="max").to(DEVICE)

train_dataset = PCLTokenDataset(train_df)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# separate loaders for F1 computation (no shuffle)
train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
dev_dataset = PCLTokenDataset(dev_df)
dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 7

def f1_on_loader(loader) -> float:
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    out = model(**batch)
            else:
                out = model(**batch)

            all_logits.append(out["paragraph_logit"].detach().float().cpu())
            all_labels.append(batch["paragraph_label"].detach().float().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)

    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    return f1_score(labels, preds, pos_label=1)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader):
        optimizer.zero_grad(set_to_none=True)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                out = model(**batch)
                loss = out["loss"]
        else:
            out = model(**batch)
            loss = out["loss"]

        if torch.isnan(loss) or torch.isinf(loss):
            raise RuntimeError("Loss became NaN/Inf. Inspect batch / masks / logits.")

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += float(loss.detach().cpu())

    train_f1 = f1_on_loader(train_eval_loader)
    dev_f1 = f1_on_loader(dev_eval_loader)

    print(
        f"Epoch {epoch} | Loss: {total_loss / len(train_loader):.4f} | "
        f"train_f1@0.5: {train_f1:.4f} | dev_f1@0.5: {dev_f1:.4f} | "
        f"backbone={MODEL_KEY} | amp={USE_AMP} | dtype={AMP_DTYPE}"
    )

/tmp/ipykernel_263150/1771181930.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))


DEVICE=cuda | backbone=albert | amp=True | amp_dtype=torch.float16 | scaler=True


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 376.69it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-base-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  7%|▋         | 35/524 [00:10<02:22,  3.44it/s]


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=32)
    model.eval()

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            logits = out["paragraph_logit"]
            all_logits.append(logits.cpu())
            all_labels.append(batch["paragraph_label"].cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    probs = 1 / (1 + np.exp(-logits))

    best_f1 = 0
    best_thresh = 0.5

    for t in np.linspace(0.1, 0.9, 81):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    return best_f1, best_thresh